In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.metrics import classification_report, accuracy_score

In [2]:
df = pd.read_csv('insurance.csv')

In [3]:
df.sample(5)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category
7,31,105.7,1.78,10.865821,True,Delhi,government_job,Medium
76,62,99.1,1.50,1.120000,False,Mysore,retired,High
91,38,119.8,1.76,28.467885,False,Bangalore,government_job,Low
55,47,75.7,1.73,24.930000,False,Delhi,unemployed,Low
98,27,101.1,1.82,28.300000,False,Kolkata,business_owner,Low


In [4]:
temp = df.copy()

In [5]:
# Feature 1: BMI
temp['bmi'] = round(temp['weight'] / (temp['height']**2),2)

In [6]:
temp.head(2)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category,bmi
0,67,119.8,1.56,2.92,False,Jaipur,retired,High,49.23
1,36,101.1,1.83,34.28,False,Chennai,freelancer,Low,30.19


In [7]:
# Feature 2: Age Group
def age_group(age):
    if age < 25:
        return "young"
    elif age < 45:
        return "adult"
    elif age < 60:
        return "middle_aged"
    else:
        return "senior"

In [8]:
temp['age_group'] = temp['age'].apply(age_group)

In [9]:
temp.sample(2)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category,bmi,age_group
13,28,93.4,1.84,11.95,False,Kolkata,freelancer,Low,27.59,adult
84,75,86.2,1.73,0.62,True,Jaipur,retired,High,28.80,senior


In [10]:
# Feature 3: Lifestyle Risk
def lifestyle_risk(row):
    if row['smoker'] and row['bmi'] > 30:
        return "high"
    elif row['smoker'] or row['bmi'] > 27:
        return "medium"
    else:
        return "low"

In [11]:
temp['lifestyle_risk'] = temp.apply(lifestyle_risk, axis=1)

In [12]:
temp.sample(4)

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category,bmi,age_group,lifestyle_risk
17,65,90.1,1.70,2.23,False,Delhi,retired,Medium,31.18,senior,medium
95,36,52.8,1.57,19.64,False,Indore,business_owner,Low,21.42,adult,low
89,24,108.8,1.89,4.00,False,Chennai,student,Low,30.46,young,medium
56,24,101.9,1.55,2.86,True,Kolkata,student,Medium,42.41,young,high


In [13]:
tier_1_cities = ["Mumbai", "Delhi", "Bangalore", "Chennai", "Kolkata", "Hyderabad", "Pune"]
tier_2_cities = [
    "Jaipur", "Chandigarh", "Indore", "Lucknow", "Patna", "Ranchi", "Visakhapatnam", "Coimbatore",
    "Bhopal", "Nagpur", "Vadodara", "Surat", "Rajkot", "Jodhpur", "Raipur", "Amritsar", "Varanasi",
    "Agra", "Dehradun", "Mysore", "Jabalpur", "Guwahati", "Thiruvananthapuram", "Ludhiana", "Nashik",
    "Allahabad", "Udaipur", "Aurangabad", "Hubli", "Belgaum", "Salem", "Vijayawada", "Tiruchirappalli",
    "Bhavnagar", "Gwalior", "Dhanbad", "Bareilly", "Aligarh", "Gaya", "Kozhikode", "Warangal",
    "Kolhapur", "Bilaspur", "Jalandhar", "Noida", "Guntur", "Asansol", "Siliguri"
]

In [14]:
# Feature 4: City Tier
def city_tier(city):
    if city in tier_1_cities:
        return 1
    elif city in tier_2_cities:
        return 2
    else:
        return 3

In [15]:
temp['city_tier'] = temp['city'].apply(city_tier)

In [17]:
temp.sample()

,age,weight,height,income_lpa,smoker,city,occupation,insurance_premium_category,bmi,age_group,lifestyle_risk,city_tier
88,54,97.4,1.76,30.0,False,Mumbai,government_job,Low,31.44,middle_aged,medium,1


In [18]:
temp.drop(columns=['age', 'weight', 'height', 'smoker', 'city'])[['income_lpa', 'occupation', 'bmi', 'age_group', 'lifestyle_risk', 'city_tier', 'insurance_premium_category']].sample(5)

,income_lpa,occupation,bmi,age_group,lifestyle_risk,city_tier,insurance_premium_category
3,3.34,student,45.54,young,high,1,Medium
36,0.53,retired,21.71,senior,low,1,Medium
41,3.87,retired,22.51,senior,low,1,Medium
50,25.23,private_job,18.87,middle_aged,medium,2,Medium
21,2.91,retired,27.38,senior,medium,2,High


In [20]:
# Select features and target
X = temp[['bmi', 'age_group', 'lifestyle_risk', 'city_tier', 'income_lpa', 'occupation']]
y = temp['insurance_premium_category']

In [21]:
X.head()

,bmi,age_group,lifestyle_risk,city_tier,income_lpa,occupation
0,49.23,senior,medium,2,2.92,retired
1,30.19,adult,medium,1,34.28,freelancer
2,21.12,adult,low,2,36.64,freelancer
3,45.54,young,high,1,3.34,student
4,24.30,senior,medium,2,3.94,retired


In [23]:
y.head()

0      High
1       Low
2       Low
3    Medium
4      High
Name: insurance_premium_category, dtype: object

In [24]:
# Define categorical and numeric features
cat_features = ['age_group', 'lifestyle_risk', 'occupation', 'city_tier']
num_features = ['bmi', 'income_lpa']

In [26]:
# create column transformer for OHE
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(), cat_features),
        ('num', 'passthrough', num_features)
    ]
)

In [27]:
# create a pipeline with preprocessing and random forest classifier
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

In [28]:
# Split data and train model
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.2,random_state=1)
pipeline.fit(X_train,y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat', OneHotEncoder(),
                                                  ['age_group',
                                                   'lifestyle_risk',
                                                   'occupation', 'city_tier']),
                                                 ('num', 'passthrough',
                                                  ['bmi', 'income_lpa'])])),
                ('classifier', RandomForestClassifier(random_state=42))])

In [29]:
# Predict and evaluate
y_pred = pipeline.predict(X_test)
accuracy_score(y_test,y_pred)

0.9

In [30]:
X_test.sample(5)

,bmi,age_group,lifestyle_risk,city_tier,income_lpa,occupation
65,37.66,middle_aged,high,2,38.07,unemployed
10,22.95,adult,medium,1,32.78,business_owner
32,31.50,middle_aged,medium,2,50.00,private_job
92,18.32,adult,medium,2,30.00,government_job
81,31.87,adult,high,2,22.19,freelancer


In [32]:
import pickle

# Save the trained pipeline using pickle
pickle_model_path = "model.pkl"
with open(pickle_model_path, 'wb') as f:
    pickle.dump(pipeline,f)